### Save to drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os, shutil

# Set your Drive backup folder
DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)

print(f"Drive mounted. Backup folder: {DRIVE_BACKUP}")

Mounted at /content/drive
Drive mounted. Backup folder: /content/drive/MyDrive/HRM_Manifolds_backup


### post every run we run this to save to drive

In [ ]:
import os, shutil, json
from datetime import datetime

project_root = "/content/HRM-Manifolds/Flight-v1"
DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

def save_to_drive(label: str):
    """Copy all important outputs to Drive."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    print(f"\nSaving {label} to Drive...")

    # What to save and where
    items = [
        # Checkpoints — largest files
        ("checkpoints", "checkpoints"),
        # Outputs — json configs and reports
        ("outputs", "outputs"),
        # Trace library — embeddings and traces
        ("trace_library", "trace_library"),
        # Logs
        ("logs", "logs"),
        # Configs — in case they were modified
        ("configs", "configs"),
    ]

    for src_rel, dst_rel in items:
        src = os.path.join(project_root, src_rel)
        dst = os.path.join(DRIVE_BACKUP, dst_rel)
        if not os.path.exists(src):
            print(f"  SKIP {src_rel} — not found")
            continue
        print(f"  Copying {src_rel}...", end=" ")
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        # Report size
        total = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(dst)
            for f in files
        )
        print(f"done ({total/1e6:.1f} MB)")

    # Write a manifest so you know what stage this backup is from
    manifest = {
        "label": label,
        "timestamp": timestamp,
        "project_root": project_root,
    }
    # Add stage status if it exists
    status_path = os.path.join(project_root, "outputs", "stage_status.json")
    if os.path.exists(status_path):
        with open(status_path) as f:
            manifest["stage_status"] = json.load(f)

    manifest_path = os.path.join(DRIVE_BACKUP, f"manifest_{timestamp}.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nManifest saved: {manifest_path}")
    print(f"Backup complete: {DRIVE_BACKUP}")

# Run the backup now
save_to_drive("stages_0_to_4_complete")


Saving stages_0_to_4_complete to Drive...
  SKIP checkpoints — not found
  SKIP outputs — not found
  SKIP trace_library — not found
  SKIP logs — not found
  SKIP configs — not found

Manifest saved: /content/drive/MyDrive/HRM_Manifolds_backup/manifest_20260401_211247.json
Backup complete: /content/drive/MyDrive/HRM_Manifolds_backup


### restore from drive

In [ ]:
# RESTORE CELL — run this after reconnecting to restore from Drive
# Run after cells 1-5 (GPU check, deps, clone, dirs, runner)

from google.colab import drive
drive.mount("/content/drive")

import os, shutil

project_root = "/content/HRM-Manifolds/Flight-v1"
DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

def restore_from_drive():
    """Restore saved outputs back to /content after a disconnect."""
    items = [
        "checkpoints",
        "outputs",
        "trace_library",
        "logs",
        "configs",
    ]

    print("Restoring from Drive...\n")
    for folder in items:
        src = os.path.join(DRIVE_BACKUP, folder)
        dst = os.path.join(project_root, folder)
        if not os.path.exists(src):
            print(f"  SKIP {folder} — not in backup")
            continue
        print(f"  Restoring {folder}...", end=" ")
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        total = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(dst)
            for f in files
        )
        print(f"done ({total/1e6:.1f} MB)")

    print("\nRestore complete. Run Cell 16 to check pipeline status.")

restore_from_drive()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restoring from Drive...

  Restoring checkpoints... done (907.9 MB)
  Restoring outputs... done (2.6 MB)
  Restoring trace_library... done (388.2 MB)
  Restoring logs... done (0.6 MB)
  SKIP configs — not in backup

Restore complete. Run Cell 16 to check pipeline status.


### Check up

In [ ]:
# Cell 1 — Runtime check and GPU verification
import subprocess
import sys
import os

def check_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader"],
            capture_output=True, text=True
        )
        print("GPU detected:")
        print(result.stdout.strip())
    except FileNotFoundError:
        raise RuntimeError(
            "No GPU found. Go to Runtime > Change runtime type "
            "and select A100 GPU before continuing."
        )

def check_python():
    version = sys.version_info
    assert version.major == 3 and version.minor >= 10, (
        f"Python 3.10+ required, got {version.major}.{version.minor}"
    )
    print(f"Python {version.major}.{version.minor}.{version.micro} OK")

def check_disk():
    result = subprocess.run(
        ["df", "-h", "/"],
        capture_output=True, text=True
    )
    print("Disk space:")
    print(result.stdout.strip())

check_gpu()
check_python()
check_disk()
print("\nRuntime OK. Proceed to Cell 2.")

GPU detected:
NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB
Python 3.12.13 OK
Disk space:
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   44G  192G  19% /

Runtime OK. Proceed to Cell 2.


### Setup dependencies

In [ ]:
# Cell 2 — Install dependencies
import subprocess
import sys

def pip_install(packages):
    for pkg in packages:
        print(f"Installing {pkg}...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pkg],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"FAILED: {pkg}")
            print(result.stderr)
            raise RuntimeError(f"Failed to install {pkg}")
        else:
            print(f"  OK: {pkg}")

packages = [
   # "torch==2.2.0",
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "accelerate>=0.27.0",
    "pyyaml",
    "tqdm",
    "numpy",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "sentencepiece",
    "protobuf",
    "einops",
]

pip_install(packages)

# Verify torch + CUDA
import torch
assert torch.cuda.is_available(), (
    "CUDA not available after install. Restart runtime and try again."
)
print(f"\ntorch {torch.__version__}")
print(f"CUDA {torch.version.cuda}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("\nDependencies OK. Proceed to Cell 3.")

Installing transformers>=4.40.0...
  OK: transformers>=4.40.0
Installing datasets>=2.18.0...
  OK: datasets>=2.18.0
Installing accelerate>=0.27.0...
  OK: accelerate>=0.27.0
Installing pyyaml...
  OK: pyyaml
Installing tqdm...
  OK: tqdm
Installing numpy...
  OK: numpy
Installing scipy...
  OK: scipy
Installing scikit-learn...
  OK: scikit-learn
Installing matplotlib...
  OK: matplotlib
Installing sentencepiece...
  OK: sentencepiece
Installing protobuf...
  OK: protobuf
Installing einops...
  OK: einops

torch 2.10.0+cu128
CUDA 12.8
Device: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB

Dependencies OK. Proceed to Cell 3.


### Repo setup

In [ ]:
# Cell 3 — Clone repo and set working directory
import os
import subprocess

# ── CONFIG ─────────────────────────────────────────────
REPO_URL   = "https://github.com/Tulsani/HRM-Manifolds.git"
REPO_NAME  = "HRM-Manifolds"
BRANCH     = "flight-v1"
COLAB_ROOT = "/content"
PROJECT_DIR = os.path.join(COLAB_ROOT, REPO_NAME)
# ────────────────────────────────────────────────────────

def clone_or_pull():
    if os.path.exists(PROJECT_DIR):
        print(f"Repo already exists at {PROJECT_DIR}")
        print("Pulling latest changes...")
        result = subprocess.run(
            ["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH],
            capture_output=True, text=True
        )
        print(result.stdout.strip())
        if result.returncode != 0:
            print("WARNING: git pull failed. Using existing code.")
            print(result.stderr)
    else:
        print(f"Cloning {REPO_URL}...")
        result = subprocess.run(
            ["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Clone failed:\n{result.stderr}\n\n"
                "Check that REPO_URL is correct and the repo is public."
            )
        print(f"Cloned to {PROJECT_DIR}")

clone_or_pull()

# Set working directory and Python path
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Verify expected structure
required_dirs = ["backbone", "data", "training", "utils", "configs"]
missing = [d for d in required_dirs if not os.path.isdir(d)]
if missing:
    print(f"WARNING: Missing directories: {missing}")
    print("Make sure Stage 0 code was committed to the repo.")
else:
    print("Project structure verified.")

print(f"\nWorking directory: {os.getcwd()}")
print("Proceed to Cell 4.")

Cloning https://github.com/Tulsani/HRM-Manifolds.git...
Cloned to /content/HRM-Manifolds
Make sure Stage 0 code was committed to the repo.

Working directory: /content/HRM-Manifolds
Proceed to Cell 4.


In [ ]:
%cd /content/HRM-Manifolds/Flight-v1/

/content/HRM-Manifolds/Flight-v1


directory setup

In [ ]:
# Cell 4 — Checkpoint and directory setup
import os
import json
from datetime import datetime

# All directories the pipeline needs
DIRS = [
    "checkpoints",
    "trace_library",
    "trace_library/logits",
    "trace_library/embeddings",
    "outputs",
    "logs",
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}/")

# Stage status tracker
STATUS_FILE = "outputs/stage_status.json"

def load_status():
    if os.path.exists(STATUS_FILE):
        with open(STATUS_FILE) as f:
            return json.load(f)
    return {
        "stage0": "pending",
        "stage1": "pending",
        "stage2": "pending",
        "stage3": "pending",
        "stage4": "pending",
        "stage5": "pending",
        "stage6": "pending",
        "last_updated": None,
    }

def save_status(status):
    status["last_updated"] = datetime.now().isoformat()
    with open(STATUS_FILE, "w") as f:
        json.dump(status, f, indent=2)

def mark_stage(stage, state):
    status = load_status()
    status[stage] = state
    save_status(status)
    print(f"Stage {stage} marked as: {state}")

def print_status():
    status = load_status()
    print("\n=== Pipeline Status ===")
    icons = {
        "pending":  "○",
        "running":  "◉",
        "complete": "✓",
        "failed":   "✗",
        "skipped":  "—",
    }
    for stage in ["stage0","stage1","stage2",
                  "stage3","stage4","stage5","stage6"]:
        icon = icons.get(status[stage], "?")
        print(f"  {icon}  {stage}: {status[stage]}")
    if status["last_updated"]:
        print(f"\n  Last updated: {status['last_updated']}")
    print()

# Checkpoint size reporter
def report_checkpoints():
    ckpt_dir = "checkpoints"
    if not os.path.exists(ckpt_dir):
        print("No checkpoints yet.")
        return
    files = sorted(os.listdir(ckpt_dir))
    if not files:
        print("No checkpoints yet.")
        return
    print("\n=== Checkpoints ===")
    total = 0
    for f in files:
        path = os.path.join(ckpt_dir, f)
        size = os.path.getsize(path) / 1e6
        total += size
        print(f"  {f:<45} {size:>8.1f} MB")
    print(f"  {'TOTAL':<45} {total:>8.1f} MB")

save_status(load_status())
print_status()
report_checkpoints()
print("Setup complete. Proceed to Cell 5.")

  checkpoints/
  trace_library/
  trace_library/logits/
  trace_library/embeddings/
  outputs/
  logs/

=== Pipeline Status ===
  ○  stage0: pending
  ○  stage1: pending
  ○  stage2: pending
  ○  stage3: pending
  ○  stage4: pending
  ○  stage5: pending
  ○  stage6: pending

  Last updated: 2026-04-03T10:53:42.840512

No checkpoints yet.
Setup complete. Proceed to Cell 5.


### stage runnner

In [ ]:
# Cell 5 — Stage runner utility
import subprocess
import sys
import os
import time
import json
from datetime import datetime

LOG_DIR = "logs"

def run_stage(
    stage_name: str,
    command: list,
    config_path: str,
    expected_outputs: list,
    force_rerun: bool = False,
):
    """
    Run a single pipeline stage.

    Parameters
    ----------
    stage_name      : e.g. "stage0"
    command         : e.g. ["python", "-m", "training.train_backbone"]
    config_path     : path to the stage yaml config
    expected_outputs: list of files that must exist after success
    force_rerun     : if True, re-run even if already complete
    """

    status = load_status()

    # Skip if already complete
    if status[stage_name] == "complete" and not force_rerun:
        print(f"\n{stage_name} already complete. Skipping.")
        print("Pass force_rerun=True to re-run.")
        return True

    # Verify config exists
    if not os.path.exists(config_path):
        raise FileNotFoundError(
            f"Config not found: {config_path}\n"
            f"Make sure the repo contains configs/ from all stages."
        )

    print(f"\n{'='*60}")
    print(f"  Running {stage_name}")
    print(f"  Command: {' '.join(command)}")
    print(f"  Config:  {config_path}")
    print(f"  Started: {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*60}\n")

    mark_stage(stage_name, "running")

    log_path = os.path.join(LOG_DIR, f"{stage_name}.log")
    start_time = time.time()

    try:
        with open(log_path, "w") as log_file:
            process = subprocess.Popen(
                command + ["--config", config_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            # Stream output live + write to log
            output_lines = []
            for line in process.stdout:
                print(line, end="")
                log_file.write(line)
                output_lines.append(line)

            process.wait()

        elapsed = time.time() - start_time

        if process.returncode != 0:
            mark_stage(stage_name, "failed")
            print(f"\n{'='*60}")
            print(f"  FAILED: {stage_name}")
            print(f"  Exit code: {process.returncode}")
            print(f"  Elapsed: {elapsed:.1f}s")
            print(f"  Log saved: {log_path}")
            print(f"{'='*60}")
            print("\nLast 30 lines of output:")
            print("".join(output_lines[-30:]))
            raise RuntimeError(
                f"{stage_name} failed with exit code {process.returncode}.\n"
                f"Full log: {log_path}\n"
                f"Fix the error above then re-run this cell."
            )

        # Verify expected outputs exist
        missing_outputs = [
            f for f in expected_outputs
            if not os.path.exists(f)
        ]
        if missing_outputs:
            mark_stage(stage_name, "failed")
            raise RuntimeError(
                f"{stage_name} exited cleanly but expected outputs "
                f"are missing:\n"
                + "\n".join(f"  - {f}" for f in missing_outputs)
                + f"\n\nCheck the log: {log_path}"
            )

        mark_stage(stage_name, "complete")
        print(f"\n{'='*60}")
        print(f"  COMPLETE: {stage_name}")
        print(f"  Elapsed: {elapsed/60:.1f} min")
        print(f"  Log: {log_path}")
        print(f"{'='*60}\n")
        return True

    except RuntimeError:
        raise

    except Exception as e:
        mark_stage(stage_name, "failed")
        raise RuntimeError(
            f"Unexpected error in {stage_name}: {e}"
        ) from e


def memory_snapshot(label=""):
    import torch
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU memory {label}: {used:.1f} / {total:.1f} GB used")

def clear_gpu_memory():
    import torch, gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU cache cleared.")

print("Stage runner loaded. Proceed to Cell 6.")

Stage runner loaded. Proceed to Cell 6.


#### rerun to reset stage

In [ ]:
import json, os

project_root = "/content/HRM-Manifolds/Flight-v1"
status_path = os.path.join(project_root, "outputs", "stage_status.json")

with open(status_path) as f:
    status = json.load(f)

# Reset stage 3 and everything after it
status["stage3"] = "complete"
status["stage4"] = "complete"
status["stage5"] = "pending"
status["stage6"] = "pending"

with open(status_path, "w") as f:
    json.dump(status, f, indent=2)

# Delete broken stage 3 checkpoints
ckpt_dir = os.path.join(project_root, "checkpoints")
deleted = []
for fname in os.listdir(ckpt_dir):
    if "stage3" in fname and fname.endswith(".pt"):
        os.remove(os.path.join(ckpt_dir, fname))
        deleted.append(fname)

print(f"Deleted: {deleted}")
print("\nCurrent status:")
for stage, state in status.items():
    if stage != "last_updated":
        icon = "✓" if state == "complete" else "○"
        print(f"  {icon}  {stage}: {state}")
print("\nReady to re-run Cell 9 (Stage 3).")

Deleted: ['expert_algebraic_stage3.pt', 'expert_number_theory_stage3.pt', 'expert_multi_step_reason_stage3.pt', 'expert_combinatorics_stage3.pt', 'expert_geometric_stage3.pt']

Current status:
  ✓  stage0: complete
  ✓  stage1: complete
  ✓  stage2: complete
  ✓  stage3: complete
  ✓  stage4: complete
  ○  stage5: pending
  ○  stage6: pending

Ready to re-run Cell 9 (Stage 3).


### Backbone pre training: stage 0

In [ ]:
# Cell 6 — Stage 0: backbone pretraining
# Expected runtime: 30–60 min on A100
# Produces: checkpoints/backbone_stage0.pt

memory_snapshot("before Stage 0")

run_stage(
    stage_name    = "stage0",
    command       = ["python", "-m", "training.train_backbone"],
    config_path   = "configs/backbone_small.yaml",
    expected_outputs = [
        "checkpoints/backbone_stage0.pt",
    ],
)

memory_snapshot("after Stage 0")
report_checkpoints()
print_status()

Streaming output truncated to the last 5000 lines.
Training backbone:   9%|▉         | 450/5000 [06:34<49:33,  1.53it/s]
                                                                     

Training backbone:  10%|█         | 500/5000 [07:07<47:14,  1.59it/s]
                                                                     

Training backbone:  11%|█         | 550/5000 [07:39<46:57,  1.58it/s]
                                                                     

Training backbone:  12%|█▏        | 600/5000 [08:11<48:23,  1.52it/s]
                                                                     

Training backbone:  13%|█▎        | 650/5000 [08:43<45:14,  1.60it/s]
                                                                     

Training backbone:  14%|█▍        | 700/5000 [09:15<49:23,  1.45it/s]
                                                                     

Training backbone:  15%|█▌        | 750/5000 [09:47<48:16,  1.47it/s]
                                 

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage0")

Saving stage0 outputs to Drive...
  OK  checkpoints/backbone_stage0.pt (906.3 MB)
  OK  logs/stage0.log (0.5 MB)
Done. Drive backup updated for stage0.


### Teacher Trace Generation : Stage 1

#### quick fixes should not be needed

In [ ]:
# Fix Cell — patch MATH dataset path in trace_generator.py
import re

filepath = "data/trace_generator.py"

with open(filepath, "r") as f:
    content = f.read()

# Show what we are replacing
old_calls = [
    line.strip() for line in content.splitlines()
    if "lighteval/MATH" in line or "hendrycks/competition_math" in line
]
print("Found these dataset references:")
for line in old_calls:
    print(f"  {line}")

# Replace lighteval/MATH with the working alternative
content = content.replace(
    'load_dataset("lighteval/MATH", "all", split="train")',
    'load_dataset("EleutherAI/hendrycks_math", "all", split="train")',
)

# Also catch any variant spellings
content = content.replace(
    "load_dataset('lighteval/MATH', 'all', split='train')",
    "load_dataset('EleutherAI/hendrycks_math', 'all', split='train')",
)

# Also patch hendrycks/competition_math if present
content = content.replace(
    'load_dataset("hendrycks/competition_math"',
    'load_dataset("EleutherAI/hendrycks_math"',
)

with open(filepath, "w") as f:
    f.write(content)

print("\nPatched. Verifying...")

# Confirm the fix landed
with open(filepath, "r") as f:
    patched = f.read()

assert "lighteval/MATH" not in patched, (
    "lighteval/MATH still present — check the file manually"
)

remaining = [
    line.strip() for line in patched.splitlines()
    if "EleutherAI/hendrycks_math" in line
]
print("New dataset references:")
for line in remaining:
    print(f"  {line}")

print("\nPatch applied. Re-run Cell 7.")

Found these dataset references:
  dataset: Dataset = load_dataset("lighteval/MATH", "all", split="train")

Patched. Verifying...
New dataset references:
  dataset: Dataset = load_dataset("EleutherAI/hendrycks_math", "all", split="train")

Patch applied. Re-run Cell 7.


In [ ]:
import os

project_root = "/content/HRM-Manifolds/Flight-v1"

# Check both files
for fname in ["trace_generator.py", "trace_generator_v1.py"]:
    fpath = os.path.join(project_root, "data", fname)
    if os.path.exists(fpath):
        with open(fpath) as f:
            content = f.read()
        has_old = "lighteval/MATH" in content
        has_new = "EleutherAI/hendrycks_math" in content
        has_fix = "min(len(valid_generated_ids)" in content
        print(f"\n{fname}:")
        print(f"  lighteval/MATH present:    {has_old}")
        print(f"  hendrycks_math present:    {has_new}")
        print(f"  score_index fix present:   {has_fix}")
    else:
        print(f"\n{fname}: NOT FOUND")

# Check which one gets imported
import subprocess
result = subprocess.run(
    ["python", "-c",
     "import sys; sys.path.insert(0, '.'); "
     "from data import trace_generator; "
     "print(trace_generator.__file__)"],
    capture_output=True, text=True,
    cwd=project_root
)
print(f"\nImported module path: {result.stdout.strip()}")
print(f"Import error: {result.stderr.strip()}" if result.stderr else "")


trace_generator.py:
  lighteval/MATH present:    False
  hendrycks_math present:    True
  score_index fix present:   False

trace_generator_v1.py:
  lighteval/MATH present:    False
  hendrycks_math present:    True
  score_index fix present:   False

Imported module path: /content/HRM-Manifolds/Flight-v1/data/trace_generator.py



In [ ]:
# Verify cell — confirm dataset loads before re-running Stage 1
from datasets import load_dataset

print("Testing EleutherAI/hendrycks_math...")
try:
    ds = load_dataset(
        "EleutherAI/hendrycks_math",
        "all",
        split="train",
        streaming=True   # streaming so we don't download the whole thing
    )
    # Just grab the first example to confirm it works
    first = next(iter(ds))
    print(f"OK. First example keys: {list(first.keys())}")
    print(f"Sample problem: {first['problem'][:80]}...")
except Exception as e:
    print(f"FAILED: {e}")
    print("\nTrying fallback: competition_math...")
    try:
        ds = load_dataset(
            "competition_math",
            split="train",
            streaming=True
        )
        first = next(iter(ds))
        print(f"Fallback OK. Keys: {list(first.keys())}")
        print("Update the patch cell to use 'competition_math' instead.")
    except Exception as e2:
        print(f"Fallback also failed: {e2}")
        print("Check HuggingFace Hub for current MATH dataset availability.")

Testing EleutherAI/hendrycks_math...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


FAILED: BuilderConfig 'all' not found. Available: ['algebra', 'counting_and_probability', 'geometry', 'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus']

Trying fallback: competition_math...
Fallback also failed: Dataset 'competition_math' doesn't exist on the Hub or cannot be accessed.
Check HuggingFace Hub for current MATH dataset availability.


Minor fixes on fly

In [ ]:
# Write the fixed trace_generator.py to the correct path

import os

# Auto-detect the actual project root
possible_roots = [
    "/content/HRM-Manifolds/Flight-v1",
    "/content/skill_distill",
    "/content/HRM-Manifolds",
]

project_root = None
for root in possible_roots:
    if os.path.exists(os.path.join(root, "data", "trace_generator.py")):
        project_root = root
        break

if project_root is None:
    # Search more broadly
    for dirpath, dirnames, filenames in os.walk("/content"):
        if "trace_generator.py" in filenames and "data" in dirpath:
            project_root = dirpath.replace("/data", "")
            break

print(f"Found project root: {project_root}")
target = os.path.join(project_root, "data", "trace_generator.py")
print(f"Will overwrite: {target}")

Found project root: /content/HRM-Manifolds/Flight-v1
Will overwrite: /content/HRM-Manifolds/Flight-v1/data/trace_generator.py


In [ ]:
import os, shutil

project_root = "/content/HRM-Manifolds/Flight-v1"
pycache = os.path.join(project_root, "data", "__pycache__")

if os.path.exists(pycache):
    shutil.rmtree(pycache)
    print(f"Cleared: {pycache}")
else:
    print("No __pycache__ found")

# Also check line 212 specifically in the file being used
filepath = os.path.join(project_root, "data", "trace_generator.py")
with open(filepath) as f:
    lines = f.readlines()

print(f"\nLine 210-215 of trace_generator.py:")
for i, line in enumerate(lines[209:215], start=210):
    print(f"  {i}: {line.rstrip()}")

Cleared: /content/HRM-Manifolds/Flight-v1/data/__pycache__

Line 210-215 of trace_generator.py:
  210: 
  211: def sample_math_examples(n_samples: int, seed: int = 42) -> list[dict[str, Any]]:
  212:     dataset: Dataset = load_dataset("lighteval/MATH", "all", split="train")
  213:     by_subject: dict[str, list[tuple[int, dict[str, Any]]]] = defaultdict(list)
  214:     for idx, example in enumerate(dataset):
  215:         subject = str(example.get("subject", "unknown"))


#### stage 1 run

In [ ]:
# Cell 7 — Stage 1: teacher trace generation
# Expected runtime: 60–120 min on A100
# (teacher inference over ~10K examples)
# Produces: trace_library/gsm8k_traces.jsonl
#           trace_library/math_traces.jsonl
#           trace_library/logits/

clear_gpu_memory()
memory_snapshot("before Stage 1")

run_stage(
    stage_name    = "stage1",
    command       = ["python", "-m", "scripts.run_trace_gen"],
    config_path   = "configs/trace_gen.yaml",
    expected_outputs = [
        "trace_library/gsm8k_traces.jsonl",
        "trace_library/math_traces.jsonl",
        "trace_library/logits/gsm8k_logits.npz",
        "trace_library/logits/math_logits.npz",
        "trace_library/skill_index.json",
    ],
)

# Quick sanity check on trace counts
import json

def count_jsonl(path):
    with open(path) as f:
        return sum(1 for _ in f)

gsm_count  = count_jsonl("trace_library/gsm8k_traces.jsonl")
math_count = count_jsonl("trace_library/math_traces.jsonl")
print(f"\nTrace counts:")
print(f"  GSM8K: {gsm_count:,} traces")
print(f"  MATH:  {math_count:,} traces")
assert gsm_count > 7000, (
    f"Expected >7000 GSM8K traces, got {gsm_count}. "
    "Stage 1 may have stopped early."
)
assert math_count > 2500, (
    f"Expected >2500 MATH traces, got {math_count}. "
    "Stage 1 may have stopped early."
)
print("Trace counts OK.")
clear_gpu_memory()
memory_snapshot("after Stage 1")
print_status()

GPU cache cleared.
GPU memory before Stage 1: 0.0 / 42.4 GB used

  Running stage1
  Command: python -m scripts.run_trace_gen
  Config:  configs/trace_gen.yaml
  Started: 03:42:19

Stage stage1 marked as: running

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 169.23it/s, Materializing param=model.norm.weight]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  Loaded algebra: 1744 examples
  Loaded counting_and_probability: 771 examples
  Loaded geometry: 870 examples
  Loaded intermediate_algebra: 1295 examples
  Loaded number_theory: 869 examples
  Loaded prealgebra: 1205 examples
  Loaded precalculus: 746 examples
  Total MATH examples: 7500
[gsm8k] processed 600 / 7473 examples
[gsm8k] processed 700 / 7473 examples
[gsm8k] processed 800 / 7473 examples
[gsm8k] processed 900 / 7473 examples
[gsm8k] processed 1000 / 7473 examples
[gsm8k] processed 1100 / 7473 examples
[gsm8

save

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage1")

Saving stage1 outputs to Drive...
  OK  trace_library (104.3 MB)
  OK  logs/stage1.log (0.1 MB)
Done. Drive backup updated for stage1.


### Skill labeling + geometry analysis : Stage 2

In [ ]:
# Cell 8 — Stage 2: skill labeling + geometry analysis
# Expected runtime: 15–30 min on A100
# Produces: outputs/skill_labels.json
#           outputs/geometry_config.json
#           outputs/stage2_report.md
#           trace_library/embeddings/

clear_gpu_memory()
memory_snapshot("before Stage 2")

run_stage(
    stage_name    = "stage2",
    command       = ["python", "-m", "scripts.run_stage2"],
    config_path   = "configs/stage2.yaml",
    expected_outputs = [
        "outputs/skill_labels.json",
        "outputs/geometry_config.json",
        "outputs/stage2_report.md",
        "trace_library/embeddings/problem_embeddings.npz",
        "trace_library/embeddings/step_embeddings.npz",
    ],
)

# Print geometry decisions
import json
with open("outputs/geometry_config.json") as f:
    geo = json.load(f)

print("\n=== Geometry decisions ===")
for skill, cfg in geo["skills"].items():
    print(f"  {skill:<22} → {cfg['geometry']:<12} "
          f"(h_dim={cfg.get('h_dim',0)}, "
          f"e_dim={cfg.get('e_dim',0)})")

# Print skill label summary
with open("outputs/skill_labels.json") as f:
    labels = json.load(f)

if "skill_counts" in labels:
    print("\n=== Skill distribution ===")
    for skill, count in labels["skill_counts"].items():
        print(f"  {skill:<22} {count:>5} examples")

# Show stage2 report
print("\n=== Stage 2 report preview ===")
with open("outputs/stage2_report.md") as f:
    lines = f.readlines()
print("".join(lines[:40]))

clear_gpu_memory()
memory_snapshot("after Stage 2")
print_status()

GPU cache cleared.
GPU memory before Stage 2: 0.0 / 42.4 GB used

  Running stage2
  Command: python -m scripts.run_stage2
  Config:  configs/stage2.yaml
  Started: 18:05:44

Stage stage2 marked as: running
Skill labels written:    outputs/skill_labels.json
Embeddings extracted:    10473 examples
Geometry analysis done:  5 skill groups analyzed
Geometry decisions:
  algebraic          product    (delta=0.19, depth=1.38)
  combinatorics      product    (delta=0.16, depth=0.58)
  geometric          product    (delta=0.18, depth=0.80)
  multi_step_reason  product    (delta=0.15, depth=1.14)
  number_theory      product    (delta=0.18, depth=1.30)
Report written:          outputs/stage2_report.md
geometry_config.json written: outputs/geometry_config.json
Ready for Stage 3.
Stage stage2 marked as: complete

  COMPLETE: stage2
  Elapsed: 4.0 min
  Log: logs/stage2.log


=== Geometry decisions ===
  algebraic              → product      (h_dim=16, e_dim=16)
  combinatorics          → product 

save

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage2")

Saving stage2 outputs to Drive...
  OK  outputs/skill_labels.json (2.6 MB)
  OK  outputs/geometry_config.json (0.0 MB)
  OK  outputs/stage2_report.md (0.0 MB)
  OK  trace_library/embeddings (283.9 MB)
  OK  logs/stage2.log (0.0 MB)
Done. Drive backup updated for stage2.


### Manifold Build Stage 3

In [ ]:
# Cell 9 — Stage 3: manifold expert construction + pretraining
# Expected runtime: 20–40 min on A100
# Produces: checkpoints/expert_{skill}_stage3.pt per skill

clear_gpu_memory()
memory_snapshot("before Stage 3")

# Build expected expert checkpoints from geometry config
import json, os

with open("outputs/geometry_config.json") as f:
    geo = json.load(f)

expected_expert_ckpts = [
    f"checkpoints/expert_{skill}_stage3.pt"
    for skill in geo["skills"].keys()
]

print("Expected expert checkpoints:")
for e in expected_expert_ckpts:
    print(f"  {e}")

run_stage(
    stage_name    = "stage3",
    command       = ["python", "-m", "scripts.run_stage3"],
    config_path   = "configs/stage3.yaml",
    expected_outputs = expected_expert_ckpts,
)

# Report expert sizes
print("\n=== Expert checkpoint sizes ===")
total = 0
for ckpt in expected_expert_ckpts:
    if os.path.exists(ckpt):
        size = os.path.getsize(ckpt) / 1e6
        total += size

        import torch
        meta = torch.load(ckpt, map_location="cpu")
        skill    = meta.get("skill", "?")
        geometry = meta.get("geometry", "?")
        loss     = meta.get("pretrain_loss_final", float("nan"))
        print(f"  {skill:<22} {geometry:<12} "
              f"{size:>6.1f} MB   loss={loss:.4f}")

print(f"\n  Total expert params: {total:.1f} MB")
clear_gpu_memory()
memory_snapshot("after Stage 3")
report_checkpoints()
print_status()

GPU cache cleared.
GPU memory before Stage 3: 0.0 / 42.4 GB used
Expected expert checkpoints:
  checkpoints/expert_algebraic_stage3.pt
  checkpoints/expert_combinatorics_stage3.pt
  checkpoints/expert_geometric_stage3.pt
  checkpoints/expert_multi_step_reason_stage3.pt
  checkpoints/expert_number_theory_stage3.pt

  Running stage3
  Command: python -m scripts.run_stage3
  Config:  configs/stage3.yaml
  Started: 19:31:51

Stage stage3 marked as: running
[algebraic] startup encode diagnostic: finite=False nan=True input_finite=True
[algebraic] step=50 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=100 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=150 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=200 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=250 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=300 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=350 L1=0.0000 L2=0.0000 L3=0.5000 L=0.2500
[algebraic] step=400 L1=0.0000 L2=0.0000 L3=0

save

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage3")

Saving stage3 outputs to Drive...
  OK  checkpoints (907.9 MB)
  OK  logs/stage3.log (0.0 MB)
Done. Drive backup updated for stage3.


#### Diagnosis

In [ ]:
import subprocess, os

project_root = "/content/HRM-Manifolds/Flight-v1"

# Check git status of hyperbolic.py
result = subprocess.run(
    ["git", "diff", "--stat", "manifolds/hyperbolic.py"],
    capture_output=True, text=True, cwd=project_root
)
print("Git diff stat:")
print(result.stdout or "  (no diff — file matches git HEAD)")

# Show what git HEAD has vs what's on disk
result2 = subprocess.run(
    ["git", "show", "HEAD:manifolds/hyperbolic.py"],
    capture_output=True, text=True, cwd=project_root
)
head_content = result2.stdout
print(f"\ngit HEAD has input_norm: {'input_norm' in head_content}")
print(f"git HEAD has LayerNorm:  {'LayerNorm' in head_content}")
print(f"git HEAD has gain=0.1:   {'gain=0.1' in head_content}")

Git diff stat:
  (no diff — file matches git HEAD)

git HEAD has input_norm: False
git HEAD has LayerNorm:  False
git HEAD has gain=0.1:   False


In [ ]:
import os, sys, importlib

project_root = "/content/HRM-Manifolds/Flight-v1"

# Check what's actually on disk
hyp_path = os.path.join(project_root, "manifolds", "hyperbolic.py")
with open(hyp_path) as f:
    content = f.read()

print("Checks on hyperbolic.py:")
print(f"  input_norm present:      {'input_norm' in content}")
print(f"  LayerNorm present:       {'LayerNorm' in content}")
print(f"  scale * 0.1 present:     {'* 0.1' in content}")
print(f"  gain=0.1 present:        {'gain=0.1' in content}")
print(f"  clamp(-10, 10) present:  {'clamp(-10' in content}")

# Clear pycache so old bytecode doesn't interfere
import shutil
pycache = os.path.join(project_root, "manifolds", "__pycache__")
if os.path.exists(pycache):
    shutil.rmtree(pycache)
    print("\nCleared manifolds/__pycache__")

# Also clear training pycache
for folder in ["training", "data", "router", "system"]:
    pc = os.path.join(project_root, folder, "__pycache__")
    if os.path.exists(pc):
        shutil.rmtree(pc)
        print(f"Cleared {folder}/__pycache__")

Checks on hyperbolic.py:
  input_norm present:      True
  LayerNorm present:       True
  scale * 0.1 present:     True
  gain=0.1 present:        True
  clamp(-10, 10) present:  True

Cleared manifolds/__pycache__
Cleared training/__pycache__
Cleared data/__pycache__
Cleared router/__pycache__
Cleared system/__pycache__


In [ ]:
# Force reimport from scratch
for mod_name in list(sys.modules.keys()):
    if "manifold" in mod_name or "expert" in mod_name:
        del sys.modules[mod_name]

sys.path.insert(0, project_root)
os.chdir(project_root)

import json, numpy as np, torch
from manifolds import build_expert

with open("outputs/geometry_config.json") as f:
    geo_cfg = json.load(f)

expert = build_expert("algebraic", geo_cfg, backbone_dim=512)
expert.train()

# Test with real embedding scale
emb = np.load("trace_library/embeddings/problem_embeddings.npz",
              allow_pickle=False)
arr = emb["gsm8k_train_0000"].astype(np.float32)
h = torch.from_numpy(arr).unsqueeze(0)

z = expert.encode(h)
print(f"encode finite: {torch.isfinite(z).all().item()}")
print(f"z norm: {z.norm():.4f}")

# Test loss
from training.contrastive_loss import compute_pretraining_loss

h_batch = torch.stack([
    torch.from_numpy(emb[k].astype(np.float32))
    for k in list(emb.files)[:8]
])
proto_ids = torch.zeros(8, dtype=torch.long)
step_embs = [torch.zeros(0, 512)] * 8

loss, metrics = compute_pretraining_loss(
    expert=expert,
    h=h_batch,
    teacher_embeddings=h_batch,
    step_embeddings=step_embs,
    prototype_ids=proto_ids,
)
print(f"\nLoss: {metrics['loss']:.4f}")
print(f"L1 proto:  {metrics['loss_proto']:.4f}")
print(f"L2 struct: {metrics['loss_struct']:.4f}")
print(f"L3 order:  {metrics['loss_order']:.4f}")

encode finite: False
z norm: nan

Loss: 0.0000
L1 proto:  0.0000
L2 struct: 0.0000
L3 order:  0.0000


In [ ]:
import os, shutil, sys, json

project_root = "/content/HRM-Manifolds/Flight-v1"

# Clear ALL pycache recursively
cleared = []
for root, dirs, files in os.walk(project_root):
    for d in dirs:
        if d == "__pycache__":
            path = os.path.join(root, d)
            shutil.rmtree(path)
            cleared.append(path)

print(f"Cleared {len(cleared)} pycache directories:")
for p in cleared:
    print(f"  {p}")

# Verify hyperbolic.py has the fix
hyp_path = os.path.join(project_root, "manifolds", "hyperbolic.py")
with open(hyp_path) as f:
    content = f.read()

checks = {
    "input_norm":     "input_norm" in content,
    "LayerNorm":      "LayerNorm" in content,
    "scale * 0.1":   "* 0.1" in content,
    "gain=0.1":       "gain=0.1" in content,
    "clamp(-10":      "clamp(-10" in content,
}
print("\nhyperbolic.py checks:")
all_ok = True
for label, result in checks.items():
    print(f"  {'OK' if result else 'FAIL'} {label}")
    if not result:
        all_ok = False

# Reset stage3 status
status_path = os.path.join(project_root, "outputs", "stage_status.json")
with open(status_path) as f:
    status = json.load(f)
status["stage3"] = "pending"
status["stage4"] = "pending"
status["stage5"] = "pending"
status["stage6"] = "pending"
with open(status_path, "w") as f:
    json.dump(status, f, indent=2)

# Delete old stage3 checkpoints
ckpt_dir = os.path.join(project_root, "checkpoints")
deleted = []
for fname in os.listdir(ckpt_dir):
    if "stage3" in fname and fname.endswith(".pt"):
        os.remove(os.path.join(ckpt_dir, fname))
        deleted.append(fname)
print(f"\nDeleted checkpoints: {deleted}")

if all_ok:
    print("\nAll checks passed. Re-run Cell 9 now.")
else:
    print("\nSome checks failed — hyperbolic.py needs updating.")

Cleared 10 pycache directories:
  /content/HRM-Manifolds/Flight-v1/training/__pycache__
  /content/HRM-Manifolds/Flight-v1/evaluation/__pycache__
  /content/HRM-Manifolds/Flight-v1/analysis/__pycache__
  /content/HRM-Manifolds/Flight-v1/reporting/__pycache__
  /content/HRM-Manifolds/Flight-v1/tests/__pycache__
  /content/HRM-Manifolds/Flight-v1/manifolds/__pycache__
  /content/HRM-Manifolds/Flight-v1/scripts/__pycache__
  /content/HRM-Manifolds/Flight-v1/calibration/__pycache__
  /content/HRM-Manifolds/Flight-v1/utils/__pycache__
  /content/HRM-Manifolds/Flight-v1/backbone/__pycache__

hyperbolic.py checks:
  OK input_norm
  OK LayerNorm
  OK scale * 0.1
  OK gain=0.1
  OK clamp(-10

Deleted checkpoints: ['expert_multi_step_reason_stage3.pt', 'expert_number_theory_stage3.pt', 'expert_geometric_stage3.pt', 'expert_combinatorics_stage3.pt', 'expert_algebraic_stage3.pt']

All checks passed. Re-run Cell 9 now.


### Router Training : Stage 4

In [ ]:
# Cell 10 — Stage 4: router training
# Expected runtime: 10–20 min on A100
# (trains on pre-extracted embeddings, no backbone forward passes)
# Produces: checkpoints/router_stage4.pt
#           outputs/router_calibration.json

clear_gpu_memory()
memory_snapshot("before Stage 4")

run_stage(
    stage_name    = "stage4",
    command       = ["python", "-m", "scripts.run_stage4"],
    config_path   = "configs/stage4.yaml",
    expected_outputs = [
        "checkpoints/router_stage4.pt",
        "checkpoints/router_best.pt",
        "outputs/router_calibration.json",
    ],
)

# Print calibration report
import json
with open("outputs/router_calibration.json") as f:
    cal = json.load(f)

print("\n=== Router calibration ===")
print(f"  Temperature:       {cal['temperature']:.3f}")
print(f"  ECE:               {cal['ece']:.4f}")
print(f"  MCE:               {cal['mce']:.4f}")
print(f"  Val accuracy:      {cal['val_accuracy']*100:.1f}%")
print("\n  Per-skill accuracy:")
for skill, acc in cal.get("per_skill_accuracy", {}).items():
    bar = "█" * int(acc * 20)
    print(f"    {skill:<22} {acc*100:>5.1f}%  {bar}")

clear_gpu_memory()
memory_snapshot("after Stage 4")
print_status()

GPU cache cleared.
GPU memory before Stage 4: 0.0 / 42.4 GB used

  Running stage4
  Command: python -m scripts.run_stage4
  Config:  configs/stage4.yaml
  Started: 20:32:21

Stage stage4 marked as: running
[router] step=100 L_cls=0.7804 L_sub=0.1404 L_cal=0.2259 L_ent=0.4425 L_total=0.9797 val_acc=0.3800
[router] per-skill val accuracy: {'algebraic': 0.36533333333333334, 'combinatorics': 0.27319587628865977, 'geometric': 0.44549763033175355, 'multi_step_reason': 0.3509933774834437, 'number_theory': 0.5401069518716578}
[router] step=200 L_cls=0.8304 L_sub=0.1874 L_cal=0.2272 L_ent=0.3899 L_total=1.0392 val_acc=0.4303
[router] per-skill val accuracy: {'algebraic': 0.38666666666666666, 'combinatorics': 0.3402061855670103, 'geometric': 0.46445497630331756, 'multi_step_reason': 0.4586092715231788, 'number_theory': 0.48128342245989303}
[router] step=300 L_cls=0.7648 L_sub=0.2175 L_cal=0.2187 L_ent=0.3546 L_total=0.9749 val_acc=0.4316
[router] per-skill val accuracy: {'algebraic': 0.41866666

save

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage4")

Saving stage4 outputs to Drive...
  OK  checkpoints/router_stage4.pt (0.5 MB)
  OK  checkpoints/router_best.pt (0.5 MB)
  OK  outputs/router_calibration.json (0.0 MB)
  OK  logs/stage4.log (0.0 MB)
Done. Drive backup updated for stage4.


### Full Distillation Stage 5

In [ ]:
# Cell 11 — Stage 5: full system distillation
# Expected runtime: 2–4 hours on A100
# This is the longest stage — do not disconnect runtime
# Produces: checkpoints/full_system_stage5.pt
#           checkpoints/system_best.pt
#           outputs/stage5_eval.json

clear_gpu_memory()
memory_snapshot("before Stage 5")

print("Stage 5 is the longest stage (2–4 hours).")
print("Make sure your Colab session is stable before running.")
print("Checkpoints are saved every 1000 steps automatically.\n")

run_stage(
    stage_name    = "stage5",
    command       = ["python", "-m", "scripts.run_stage5"],
    config_path   = "configs/stage5.yaml",
    expected_outputs = [
        "checkpoints/full_system_stage5.pt",
        "checkpoints/system_best.pt",
        "outputs/stage5_eval.json",
    ],
)

# Print Stage 5 eval results
import json
with open("outputs/stage5_eval.json") as f:
    results = json.load(f)

print("\n=== Stage 5 evaluation ===")
print(f"  GSM8K val accuracy:    "
      f"{results.get('gsm8k_val_accuracy', 0)*100:.1f}%")
print(f"  MATH-500 accuracy:     "
      f"{results.get('math500_accuracy', 0)*100:.1f}%")
print(f"  Abstain rate:          "
      f"{results.get('abstain_rate', 0)*100:.1f}%")
print(f"  Avg router confidence: "
      f"{results.get('avg_router_confidence', 0):.3f}")

print("\n  Per-skill (GSM8K):")
for skill, acc in results.get("per_skill_gsm8k", {}).items():
    bar = "█" * int(acc * 20)
    print(f"    {skill:<22} {acc*100:>5.1f}%  {bar}")

clear_gpu_memory()
memory_snapshot("after Stage 5")
report_checkpoints()
print_status()

GPU cache cleared.
GPU memory before Stage 5: 0.0 / 42.4 GB used
Stage 5 is the longest stage (2–4 hours).
Make sure your Colab session is stable before running.
Checkpoints are saved every 1000 steps automatically.


  Running stage5
  Command: python -m scripts.run_stage5
  Config:  configs/stage5.yaml
  Started: 10:54:32

Stage stage5 marked as: running
[distill] step=100 phase=phase1 total=5.9006 l_task=10.3222 l_kd=4.2668 l_trace=0.0914 l_geom=0.0625 l_proto=0.0000 l_diff=0.1250 l_sep=0.0000 active_skill=combinatorics grad_norm=0.2189
[distill] router distribution: {'algebraic': 0.09473146498203278, 'combinatorics': 0.2321135401725769, 'geometric': 0.281234472990036, 'multi_step_reason': 0.2770710289478302, 'number_theory': 0.11484949290752411}
[distill] step=200 phase=phase1 total=2.4880 l_task=6.8047 l_kd=3.9454 l_trace=0.0442 l_geom=0.1042 l_proto=0.0000 l_diff=0.2083 l_sep=0.0000 active_skill=algebraic grad_norm=0.2105
[distill] router distribution: {'algebraic': 0.19628289341

RuntimeError: stage5 failed with exit code 1.
Full log: logs/stage5.log
Fix the error above then re-run this cell.

In [ ]:
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["manifold", "expert"]):
        del sys.modules[mod]

import torch, json
from manifolds import build_expert

with open("outputs/geometry_config.json") as f:
    geo = json.load(f)

expert = build_expert("algebraic", geo, backbone_dim=512)
expert.train()

nan_params = [n for n, p in expert.named_parameters()
              if not torch.isfinite(p).all()]
print(f"NaN params in fresh expert: {len(nan_params)}")

h = torch.randn(4, 512) * 0.1
z = expert.encode(h)
print(f"encode finite: {torch.isfinite(z).all().item()}")

loss = z.mean()
loss.backward()

nan_grads = [n for n, p in expert.named_parameters()
             if p.grad is not None and not torch.isfinite(p.grad).all()]
print(f"NaN gradient params: {len(nan_grads)}")

if not nan_grads:
    print("\nFresh experts are clean. Re-run Cell 11.")
else:
    print(f"\nStill NaN in gradients: {nan_grads}")

NaN params in fresh expert: 0
encode finite: True
NaN gradient params: 0

Fresh experts are clean. Re-run Cell 11.


save

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage5")

### Finetuning Callibration Stage 6

In [ ]:
# Cell 12 — Stage 6: fine-tuning + calibration + final eval
# Expected runtime: 30–60 min on A100
# Produces: checkpoints/final.pt
#           outputs/final_report.md
#           outputs/stage6_eval.json

clear_gpu_memory()
memory_snapshot("before Stage 6")

run_stage(
    stage_name    = "stage6",
    command       = ["python", "-m", "scripts.run_stage6"],
    config_path   = "configs/stage6.yaml",
    expected_outputs = [
        "checkpoints/final.pt",
        "outputs/final_report.md",
        "outputs/stage6_eval.json",
    ],
)

# Print final eval results
import json
with open("outputs/stage6_eval.json") as f:
    results = json.load(f)

print("\n" + "="*60)
print("  FINAL RESULTS")
print("="*60)

main = results.get("main_results", {})
print(f"\n  GSM8K test accuracy:   {main.get('gsm8k_test', 0)*100:.1f}%")
print(f"  MATH-500 accuracy:     {main.get('math500', 0)*100:.1f}%")
print(f"  Abstain rate:          {main.get('abstain_rate', 0)*100:.1f}%")
print(f"  Avg router confidence: {main.get('avg_router_conf', 0):.3f}")

print("\n  Baselines:")
baselines = results.get("baselines", {})
print(f"    Backbone only:       "
      f"GSM8K {baselines.get('backbone_gsm8k',0)*100:.1f}%  "
      f"MATH {baselines.get('backbone_math500',0)*100:.1f}%")
print(f"    Single Euclidean:    "
      f"GSM8K {baselines.get('euclidean_gsm8k',0)*100:.1f}%  "
      f"MATH {baselines.get('euclidean_math500',0)*100:.1f}%")
print(f"    Full system:         "
      f"GSM8K {main.get('gsm8k_test',0)*100:.1f}%  "
      f"MATH {main.get('math500',0)*100:.1f}%")

print("\n  Ablations (delta from full system):")
ablations = results.get("ablations", {})
for name, delta in ablations.items():
    gsm_d = delta.get("gsm8k_delta", 0) * 100
    mat_d = delta.get("math500_delta", 0) * 100
    print(f"    {name:<25} GSM8K {gsm_d:+.1f}%  "
          f"MATH {mat_d:+.1f}%")

print("\n  Calibration:")
cal = results.get("calibration", {})
print(f"    ECE before: {cal.get('ece_before', 0):.4f}")
print(f"    ECE after:  {cal.get('ece_after', 0):.4f}")
print(f"    Abstain threshold: {cal.get('abstain_threshold', 0):.2f}")

clear_gpu_memory()
memory_snapshot("after Stage 6")
report_checkpoints()
print_status()

save

In [ ]:
# Quick save after each stage — run this immediately after
# a stage completes before moving to the next one

def quick_save(stage: str):
    import os, shutil, json
    from datetime import datetime

    project_root = "/content/HRM-Manifolds/Flight-v1"
    DRIVE_BACKUP = "/content/drive/MyDrive/HRM_Manifolds_backup"

    # Only copy what changed in this stage
    stage_outputs = {
        "stage0": ["checkpoints/backbone_stage0.pt", "logs/stage0.log"],
        "stage1": ["trace_library", "logs/stage1.log"],
        "stage2": ["outputs/skill_labels.json",
                   "outputs/geometry_config.json",
                   "outputs/stage2_report.md",
                   "trace_library/embeddings",
                   "logs/stage2.log"],
        "stage3": ["checkpoints",   # all expert_*_stage3.pt
                   "logs/stage3.log"],
        "stage4": ["checkpoints/router_stage4.pt",
                   "checkpoints/router_best.pt",
                   "outputs/router_calibration.json",
                   "logs/stage4.log"],
        "stage5": ["checkpoints/full_system_stage5.pt",
                   "checkpoints/system_best.pt",
                   "outputs/stage5_eval.json",
                   "logs/stage5.log"],
        "stage6": ["checkpoints/final.pt",
                   "outputs/final_report.md",
                   "outputs/stage6_eval.json",
                   "logs/stage6.log"],
    }

    paths = stage_outputs.get(stage, [])
    print(f"Saving {stage} outputs to Drive...")

    for rel_path in paths:
        src = os.path.join(project_root, rel_path)
        dst = os.path.join(DRIVE_BACKUP, rel_path)
        if not os.path.exists(src):
            print(f"  SKIP {rel_path} — not found")
            continue
        os.makedirs(os.path.dirname(dst) if not os.path.isdir(src) else dst, exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        size = (
            sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dst) for f in fs)
            if os.path.isdir(dst)
            else os.path.getsize(dst)
        )
        print(f"  OK  {rel_path} ({size/1e6:.1f} MB)")

    # Always update the status file
    status_src = os.path.join(project_root, "outputs", "stage_status.json")
    status_dst = os.path.join(DRIVE_BACKUP, "outputs", "stage_status.json")
    if os.path.exists(status_src):
        os.makedirs(os.path.dirname(status_dst), exist_ok=True)
        shutil.copy2(status_src, status_dst)

    print(f"Done. Drive backup updated for {stage}.")

# Call like this after each stage:
quick_save("stage6")

Saving stage6 outputs to Drive...
  SKIP checkpoints/final.pt — not found
  SKIP outputs/final_report.md — not found
  SKIP outputs/stage6_eval.json — not found
  SKIP logs/stage6.log — not found
Done. Drive backup updated for stage6.


Final Report

In [ ]:
# Cell 13 — Print final report
with open("outputs/final_report.md") as f:
    report = f.read()
print(report)

Inference Test

In [ ]:
# Cell 14 — Quick inference test
# Verify the final checkpoint works end-to-end

import sys
sys.path.insert(0, "/content/skill_distill")

from system.inference import SkillDistillInference

print("Loading final checkpoint...")
model = SkillDistillInference(
    checkpoint_path = "checkpoints/final.pt",
    device          = "cuda",
)
print("Loaded.\n")

test_problems = [
    "If a train travels at 60 mph for 2.5 hours, how far does it go?",
    "A store sells apples for $0.50 each. If Sarah buys 12 apples "
    "and pays with a $10 bill, how much change does she get?",
    "Find all integer solutions to x^2 - 5x + 6 = 0.",
    "A rectangle has a perimeter of 36 cm. If the length is twice "
    "the width, what is the area?",
    "How many ways can 5 people be seated in a row if two specific "
    "people must not sit next to each other?",
]

print("="*60)
for i, problem in enumerate(test_problems, 1):
    print(f"\nProblem {i}: {problem}")
    result = model.solve(problem, show_routing=True)
    print(f"Answer:      {result.answer}")
    print(f"Skill used:  {result.skill_used}")
    print(f"Confidence:  {result.router_conf:.3f}")
    print(f"Abstained:   {result.abstained}")
    print(f"Routing:     "
          + ", ".join(f"{k}={v:.2f}"
                      for k,v in result.routing_dist.items()
                      if v > 0.05))
    print("-"*60)